# Dengue Surge Prediction (PS 08)
### Dataset: Dengue Prediction (Supervised) — Kaggle
> **602 records · 26 columns** — Temperature, Humidity, Rainfall, Wind, Solar Radiation → **Dengue Case Count**

## STEP 1: Install Libraries

In [ ]:
!pip install xgboost pandas matplotlib seaborn scikit-learn --quiet

: 

## STEP 2: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

## STEP 3: Load Dataset

> Upload **`final.csv`** from the Kaggle dataset before running.
> Dataset link: https://www.kaggle.com/datasets/siddhvr/dengue-prediction

In [ ]:
# Upload final.csv via Colab file picker first:
# from google.colab import files
# uploaded = files.upload()

data = pd.read_csv('final.csv')

print(f"Dataset loaded successfully!")
print(f"Shape: {data.shape}  |  Rows: {data.shape[0]}, Columns: {data.shape[1]}")
data.head()

## STEP 4: Exploratory Data Analysis (EDA)

In [ ]:
print("=== Column Names ===")
print(list(data.columns))
print()
print("=== Target Column (cases) - Basic Stats ===")
print(data['cases'].describe())
print()
print("=== Missing Values ===")
print(data.isnull().sum()[data.isnull().sum() > 0])

In [ ]:
# Distribution of dengue cases
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(data['cases'], bins=30, color='#2196F3', edgecolor='white', alpha=0.85)
plt.title('Distribution of Dengue Cases', fontweight='bold')
plt.xlabel('Cases')
plt.ylabel('Frequency')
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
data['labels'].value_counts().plot(kind='bar', color=['#4CAF50','#FF9800','#F44336'], edgecolor='white')
plt.title('Risk Label Distribution', fontweight='bold')
plt.xlabel('Risk Level')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap — numeric columns only
plt.figure(figsize=(14, 9))
num_cols = data.select_dtypes(include=[np.number]).drop(columns=['serial']).corr()
mask = np.triu(np.ones_like(num_cols, dtype=bool))
sns.heatmap(num_cols, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.4,
            annot_kws={'size': 7}, cbar_kws={'shrink': 0.75})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## STEP 5: Data Preprocessing

In [ ]:
# Drop non-numeric / identifier columns
drop_cols = ['serial', 'conditions', 'stations', 'labels', 'snow', 'snowdepth']
df = data.drop(columns=[c for c in drop_cols if c in data.columns])

# Fill any missing values
df = df.fillna(method='ffill').fillna(df.median(numeric_only=True))

print(f"Clean dataset shape: {df.shape}")
print(f"Remaining columns: {list(df.columns)}")
df.head()

## STEP 6: Feature Engineering (IMPORTANT)

In [ ]:
# Lag features — previous period case counts
df['lag_1_cases'] = df['cases'].shift(1)
df['lag_2_cases'] = df['cases'].shift(2)

# Rolling averages (7-day window)
df['roll7_cases'] = df['cases'].rolling(7).mean()
df['roll7_temp']  = df['temp'].rolling(7).mean()
df['roll7_humid'] = df['humidity'].rolling(7).mean()

# Climate Risk Index — composite dengue-weather score
df['climate_risk_idx'] = (
    0.35 * df['humidity'] +
    0.30 * df['temp'] +
    0.20 * df['precip'] +
    0.15 * df['uvindex']
)

# Temp range
df['temp_range'] = df['tempmax'] - df['tempmin']

# Drop NAs introduced by lag / rolling
df = df.dropna().reset_index(drop=True)

print(f"After feature engineering shape: {df.shape}")
print(f"New features: lag_1_cases, lag_2_cases, roll7_cases, roll7_temp, roll7_humid, climate_risk_idx, temp_range")
df.head()

## STEP 7: Train-Test Split

In [ ]:
feature_cols = [c for c in df.columns if c != 'cases']
X = df[feature_cols]
y = df['cases']

# shuffle=False -> preserve time-series order
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=False
)

print(f"Training samples : {X_train.shape[0]}")
print(f"Testing  samples : {X_test.shape[0]}")
print(f"Features used    : {X_train.shape[1]}")

## STEP 8: Train Model (XGBoost)

In [ ]:
model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.85,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    random_state=42,
    verbosity=0
)

model.fit(X_train, y_train,
          eval_set=[(X_test, y_test)],
          verbose=False)

print("Model trained successfully!")

## STEP 9: Predictions

In [ ]:
predictions = model.predict(X_test)
predictions = np.clip(predictions, 0, None)

results_df = pd.DataFrame({
    'Actual Cases':    y_test.values.astype(int),
    'Predicted Cases': predictions.round(0).astype(int),
    'Difference':      (predictions - y_test.values).round(0).astype(int)
})

print("Sample Predictions (first 10 rows):")
print(results_df.head(10).to_string(index=False))

## STEP 10: Model Evaluation

In [ ]:
mape = mean_absolute_percentage_error(y_test, predictions)
mae  = mean_absolute_error(y_test, predictions)
r2   = r2_score(y_test, predictions)

print("=" * 45)
print(f"  MAPE     : {mape * 100:.2f} %")
print(f"  MAE      : {mae:.0f} cases")
print(f"  R2 Score : {r2:.4f}")
print(f"  Accuracy : {(1 - mape) * 100:.2f} %")
print("=" * 45)

## STEP 11: Graph — Actual vs Predicted (VERY IMPORTANT FOR JUDGES)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Line chart
ax1 = axes[0]
idx = range(len(y_test))
ax1.plot(idx, y_test.values,  label='Actual Cases',    color='#2196F3', linewidth=2, marker='o', markersize=4)
ax1.plot(idx, predictions,    label='Predicted Cases', color='#FF5722', linewidth=2, marker='x', markersize=4, linestyle='--')
ax1.fill_between(idx, y_test.values, predictions, alpha=0.12, color='purple')
ax1.set_title('Actual vs Predicted Dengue Cases', fontsize=13, fontweight='bold')
ax1.set_xlabel('Sample Index')
ax1.set_ylabel('Dengue Cases')
ax1.legend()
ax1.grid(alpha=0.3)

# Scatter plot
ax2 = axes[1]
ax2.scatter(y_test.values, predictions, alpha=0.55, color='#9C27B0', edgecolors='white', s=55)
lim_min = min(y_test.min(), predictions.min())
lim_max = max(y_test.max(), predictions.max())
ax2.plot([lim_min, lim_max], [lim_min, lim_max], 'r--', linewidth=1.8, label='Perfect Prediction')
ax2.set_title('Predicted vs Actual (Scatter)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Actual Cases')
ax2.set_ylabel('Predicted Cases')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle('Dengue Surge Prediction — XGBoost Model Performance', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## STEP 12: Risk Classification

In [ ]:
def risk_level(x):
    if x < 3000:
        return "Low"
    elif x < 6000:
        return "Medium"
    elif x < 10000:
        return "High"
    else:
        return "Critical"

risk = [risk_level(p) for p in predictions]

print(f"{'#':<5} {'Predicted Cases':<18} {'Risk Level'}")
print("-" * 38)
for i, (p, r) in enumerate(zip(predictions, risk)):
    print(f"{i+1:<5} {p:<18.0f} {r}")

## STEP 13: Feature Importance (BONUS)

In [ ]:
importance_df = pd.DataFrame({
    'Feature':    feature_cols,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False).head(14)

plt.figure(figsize=(10, 6))
colors = plt.cm.RdYlGn_r(np.linspace(0.15, 0.9, len(importance_df)))
bars = plt.barh(importance_df['Feature'], importance_df['Importance'], color=colors)
plt.xlabel('Feature Importance Score', fontsize=12)
plt.title('Top Features Driving Dengue Surge Prediction', fontsize=13, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)

for bar, val in zip(bars, importance_df['Importance']):
    plt.text(bar.get_width() + 0.0005, bar.get_y() + bar.get_height() / 2,
             f'{val:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

## STEP 14: Risk Distribution (BONUS)

In [ ]:
risk_counts = pd.Series(risk).value_counts()
risk_order  = ['Low', 'Medium', 'High', 'Critical']
risk_counts = risk_counts.reindex([r for r in risk_order if r in risk_counts.index], fill_value=0)

palette = {'Low': '#4CAF50', 'Medium': '#FFC107', 'High': '#FF9800', 'Critical': '#F44336'}
colors  = [palette[r] for r in risk_counts.index]

plt.figure(figsize=(8, 5))
bars = plt.bar(risk_counts.index, risk_counts.values, color=colors, edgecolor='white', linewidth=1.5, width=0.55)
plt.title('Dengue Risk Level Distribution of Predictions', fontsize=13, fontweight='bold')
plt.xlabel('Risk Level', fontsize=12)
plt.ylabel('Number of Predictions', fontsize=12)
plt.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, risk_counts.values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
             str(val), ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

print("All steps complete! Model is ready for hackathon demo.")